# Generate Enhanced Table Descriptions for World Bank Data

This notebook analyzes each table in the `worldbank` schema and generates structured markdown descriptions
that help Genie Space understand the coverage, quality, and best usage patterns for each dataset.

**Output**: A DataFrame with proposed descriptions for review before applying to tables.

In [ ]:
%pip install pandas tqdm requests --quiet

In [ ]:
# Configuration
CATALOG = "main_catalog"
SCHEMA = "worldbank"

# Maximum description length (Databricks SQL comment limit)
MAX_DESCRIPTION_LENGTH = 4000

# Limit tables to process (None for all, or set a number for testing)
TABLE_LIMIT = None

In [ ]:
import pandas as pd
import requests
from tqdm.auto import tqdm
from typing import Dict, List, Optional, Tuple
from dataclasses import dataclass
from datetime import datetime

In [ ]:
# World Bank region mapping for countries
# This maps country codes to regions for regional coverage analysis
REGION_MAPPING = {
    # Major regions (aggregates in WB data)
    'EAS': 'East Asia & Pacific',
    'ECS': 'Europe & Central Asia', 
    'LCN': 'Latin America & Caribbean',
    'MEA': 'Middle East & North Africa',
    'NAC': 'North America',
    'SAS': 'South Asia',
    'SSF': 'Sub-Saharan Africa',
}

# Countries by region (simplified mapping for major economies)
COUNTRY_TO_REGION = {
    # North America
    'USA': 'North America', 'CAN': 'North America', 'MEX': 'Latin America & Caribbean',
    # Europe
    'GBR': 'Europe & Central Asia', 'DEU': 'Europe & Central Asia', 'FRA': 'Europe & Central Asia',
    'ITA': 'Europe & Central Asia', 'ESP': 'Europe & Central Asia', 'NLD': 'Europe & Central Asia',
    # Asia Pacific
    'CHN': 'East Asia & Pacific', 'JPN': 'East Asia & Pacific', 'KOR': 'East Asia & Pacific',
    'AUS': 'East Asia & Pacific', 'IDN': 'East Asia & Pacific', 'THA': 'East Asia & Pacific',
    # South Asia
    'IND': 'South Asia', 'PAK': 'South Asia', 'BGD': 'South Asia',
    # Africa
    'NGA': 'Sub-Saharan Africa', 'ZAF': 'Sub-Saharan Africa', 'KEN': 'Sub-Saharan Africa',
    'EGY': 'Middle East & North Africa', 'SAU': 'Middle East & North Africa',
    # Latin America
    'BRA': 'Latin America & Caribbean', 'ARG': 'Latin America & Caribbean', 'COL': 'Latin America & Caribbean',
}

In [ ]:
@dataclass
class TableStats:
    """Statistics about a World Bank data table."""
    table_name: str
    indicator_id: str
    original_description: str
    
    # Coverage metrics
    total_records: int
    distinct_countries: int
    total_possible_countries: int  # 266 in World Bank
    min_year: int
    max_year: int
    year_span: int
    
    # Density metrics
    theoretical_max_records: int  # countries * years
    overall_density: float  # actual / theoretical
    
    # Year-based coverage
    coverage_by_decade: Dict[str, float]  # decade -> % countries with data
    best_coverage_period: str
    sparse_periods: List[str]
    
    # Regional coverage
    coverage_by_region: Dict[str, float]
    
    # Value statistics
    value_min: float
    value_max: float
    value_mean: float
    
    # Flags
    is_sparse: bool  # overall density < 50%
    has_recent_data: bool  # has data from last 5 years
    is_historical_only: bool  # no data after 2015

In [ ]:
def get_table_list() -> List[str]:
    """Get list of all tables in the worldbank schema."""
    tables_df = spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}")
    tables = [row.tableName for row in tables_df.collect()]
    return tables

def get_original_description(table_name: str) -> str:
    """Get the current description/comment for a table."""
    try:
        desc_df = spark.sql(f"DESCRIBE TABLE EXTENDED {CATALOG}.{SCHEMA}.{table_name}")
        for row in desc_df.collect():
            if row.col_name == 'Comment':
                return row.data_type or ""
    except:
        pass
    return ""

def table_name_to_indicator_id(table_name: str) -> str:
    """Convert sanitized table name back to indicator ID format.
    
    e.g., 'sp_pop_totl' -> 'SP.POP.TOTL'
    """
    # This is approximate - the original ID used dots
    return table_name.upper().replace('_', '.')

In [ ]:
def analyze_table(table_name: str) -> Optional[TableStats]:
    """Analyze a table and return comprehensive statistics."""
    full_table_name = f"{CATALOG}.{SCHEMA}.{table_name}"
    
    try:
        # Basic counts
        stats_df = spark.sql(f"""
            SELECT 
                COUNT(*) as total_records,
                COUNT(DISTINCT country_code) as distinct_countries,
                MIN(year) as min_year,
                MAX(year) as max_year,
                MIN(value) as value_min,
                MAX(value) as value_max,
                AVG(value) as value_mean
            FROM {full_table_name}
        """).collect()[0]
        
        total_records = stats_df.total_records
        if total_records == 0:
            return None
            
        distinct_countries = stats_df.distinct_countries
        min_year = stats_df.min_year
        max_year = stats_df.max_year
        year_span = max_year - min_year + 1
        
        # Theoretical maximum
        total_possible_countries = 266
        theoretical_max = distinct_countries * year_span
        overall_density = total_records / theoretical_max if theoretical_max > 0 else 0
        
        # Coverage by decade
        decade_df = spark.sql(f"""
            SELECT 
                CONCAT(FLOOR(year / 10) * 10, 's') as decade,
                COUNT(DISTINCT country_code) as countries_with_data,
                COUNT(DISTINCT year) as years_with_data
            FROM {full_table_name}
            GROUP BY FLOOR(year / 10) * 10
            ORDER BY decade
        """).collect()
        
        coverage_by_decade = {}
        for row in decade_df:
            # Coverage = countries with data in this decade / total distinct countries
            coverage_by_decade[row.decade] = row.countries_with_data / distinct_countries
        
        # Find best and sparse periods
        best_coverage_period = max(coverage_by_decade.items(), key=lambda x: x[1])[0] if coverage_by_decade else "N/A"
        sparse_periods = [decade for decade, cov in coverage_by_decade.items() if cov < 0.5]
        
        # Coverage by region (using available country codes)
        country_df = spark.sql(f"""
            SELECT DISTINCT country_code
            FROM {full_table_name}
        """).collect()
        
        countries_in_table = set(row.country_code for row in country_df)
        
        coverage_by_region = {}
        for region in set(COUNTRY_TO_REGION.values()):
            countries_in_region = [c for c, r in COUNTRY_TO_REGION.items() if r == region]
            if countries_in_region:
                covered = sum(1 for c in countries_in_region if c in countries_in_table)
                coverage_by_region[region] = covered / len(countries_in_region)
        
        # Check for recent data
        current_year = datetime.now().year
        has_recent_data = max_year >= current_year - 5
        is_historical_only = max_year < 2015
        
        # Get original description
        original_description = get_original_description(table_name)
        
        return TableStats(
            table_name=table_name,
            indicator_id=table_name_to_indicator_id(table_name),
            original_description=original_description,
            total_records=total_records,
            distinct_countries=distinct_countries,
            total_possible_countries=total_possible_countries,
            min_year=min_year,
            max_year=max_year,
            year_span=year_span,
            theoretical_max_records=theoretical_max,
            overall_density=overall_density,
            coverage_by_decade=coverage_by_decade,
            best_coverage_period=best_coverage_period,
            sparse_periods=sparse_periods,
            coverage_by_region=coverage_by_region,
            value_min=float(stats_df.value_min) if stats_df.value_min else 0,
            value_max=float(stats_df.value_max) if stats_df.value_max else 0,
            value_mean=float(stats_df.value_mean) if stats_df.value_mean else 0,
            is_sparse=overall_density < 0.5,
            has_recent_data=has_recent_data,
            is_historical_only=is_historical_only,
        )
        
    except Exception as e:
        print(f"Error analyzing {table_name}: {e}")
        return None

In [ ]:
def format_number(n: float) -> str:
    """Format large numbers for readability."""
    if n >= 1_000_000_000:
        return f"{n/1_000_000_000:.1f}B"
    elif n >= 1_000_000:
        return f"{n/1_000_000:.1f}M"
    elif n >= 1_000:
        return f"{n/1_000:.1f}K"
    else:
        return f"{n:.2f}"

def truncate_description(text: str, max_length: int = 600) -> str:
    """Truncate description to max length, ending at sentence boundary if possible."""
    if len(text) <= max_length:
        return text
    
    # Try to end at a sentence
    truncated = text[:max_length]
    last_period = truncated.rfind('.')
    if last_period > max_length * 0.5:  # Only if we keep at least half
        return truncated[:last_period + 1]
    
    return truncated.rstrip() + "..."

In [ ]:
def generate_description(stats: TableStats) -> str:
    """Generate a structured markdown description for a table."""
    
    lines = []
    
    # Title - use indicator ID
    lines.append(f"**{stats.indicator_id}**")
    lines.append("")
    
    # Original description (truncated)
    if stats.original_description:
        desc = truncate_description(stats.original_description, 600)
        lines.append(desc)
        lines.append("")
    
    lines.append("---")
    lines.append("")
    
    # Coverage section
    lines.append("### Coverage")
    lines.append("")
    lines.append("| Metric | Value |")
    lines.append("|--------|-------|")
    lines.append(f"| Countries | {stats.distinct_countries} of {stats.total_possible_countries} ({stats.distinct_countries*100//stats.total_possible_countries}%) |")
    lines.append(f"| Years | {stats.min_year} - {stats.max_year} |")
    lines.append(f"| Records | {stats.total_records:,} |")
    lines.append("")
    
    # Data Density section
    lines.append("### Data Density")
    lines.append("")
    density_pct = int(stats.overall_density * 100)
    lines.append(f"- **Overall**: {density_pct}% complete")
    lines.append(f"- **Best period**: {stats.best_coverage_period}")
    
    if stats.sparse_periods:
        lines.append(f"- **Sparse periods**: {', '.join(stats.sparse_periods)} (<50% coverage)")
    lines.append("")
    
    # Decade breakdown
    if stats.coverage_by_decade:
        lines.append("### Coverage by Decade")
        lines.append("")
        lines.append("| Decade | Coverage |")
        lines.append("|--------|----------|")
        for decade in sorted(stats.coverage_by_decade.keys()):
            cov = stats.coverage_by_decade[decade]
            bar = "█" * int(cov * 10) + "░" * (10 - int(cov * 10))
            lines.append(f"| {decade} | {bar} {int(cov*100)}% |")
        lines.append("")
    
    # Regional coverage (if we have data)
    if stats.coverage_by_region:
        lines.append("### Regional Coverage")
        lines.append("")
        lines.append("| Region | Coverage |")
        lines.append("|--------|----------|")
        for region in sorted(stats.coverage_by_region.keys()):
            cov = stats.coverage_by_region[region]
            lines.append(f"| {region} | {int(cov*100)}% |")
        lines.append("")
    
    # Value range
    lines.append("### Value Range")
    lines.append("")
    lines.append(f"- **Min**: {format_number(stats.value_min)}")
    lines.append(f"- **Max**: {format_number(stats.value_max)}")
    lines.append(f"- **Mean**: {format_number(stats.value_mean)}")
    lines.append("")
    
    # Usage guidance
    lines.append("### Usage Guidance")
    lines.append("")
    
    if stats.has_recent_data:
        lines.append(f"- ✓ Has recent data (up to {stats.max_year})")
    else:
        lines.append(f"- ⚠ No recent data (latest: {stats.max_year})")
    
    if stats.is_sparse:
        lines.append("- ⚠ Sparse dataset - many country/year combinations missing")
    else:
        lines.append("- ✓ Good data density for cross-country comparisons")
    
    if stats.is_historical_only:
        lines.append("- ⚠ Historical data only - not suitable for current analysis")
    
    if stats.sparse_periods:
        lines.append(f"- ⚠ Limited coverage before {stats.sparse_periods[-1]}")
    
    # Best use case
    if stats.overall_density > 0.7 and stats.has_recent_data:
        lines.append("- ✓ Suitable for global trend analysis")
    elif stats.best_coverage_period:
        lines.append(f"- Best for analysis in the {stats.best_coverage_period}")
    
    result = "\n".join(lines)
    
    # Ensure we don't exceed max length
    if len(result) > MAX_DESCRIPTION_LENGTH:
        # Truncate by removing optional sections
        lines_truncated = []
        in_optional = False
        for line in lines:
            if line.startswith("### Coverage by Decade") or line.startswith("### Regional Coverage"):
                in_optional = True
            elif line.startswith("### ") and in_optional:
                in_optional = False
            
            if not in_optional:
                lines_truncated.append(line)
        
        result = "\n".join(lines_truncated)
    
    return result

In [ ]:
# Get all tables
tables = get_table_list()
print(f"Found {len(tables)} tables in {CATALOG}.{SCHEMA}")

if TABLE_LIMIT:
    tables = tables[:TABLE_LIMIT]
    print(f"Limited to first {TABLE_LIMIT} tables")

In [ ]:
# Analyze all tables and generate descriptions
results = []

for table_name in tqdm(tables, desc="Analyzing tables"):
    stats = analyze_table(table_name)
    if stats:
        description = generate_description(stats)
        results.append({
            'table_name': table_name,
            'indicator_id': stats.indicator_id,
            'records': stats.total_records,
            'countries': stats.distinct_countries,
            'year_range': f"{stats.min_year}-{stats.max_year}",
            'density': f"{int(stats.overall_density * 100)}%",
            'is_sparse': stats.is_sparse,
            'has_recent_data': stats.has_recent_data,
            'proposed_description': description,
            'description_length': len(description),
        })

print(f"\nAnalyzed {len(results)} tables successfully")

In [ ]:
# Create DataFrame with results
df_results = pd.DataFrame(results)
print(f"Results DataFrame shape: {df_results.shape}")
print(f"\nSummary:")
print(f"  Sparse tables (<50% density): {df_results['is_sparse'].sum()}")
print(f"  Tables with recent data: {df_results['has_recent_data'].sum()}")
print(f"  Average description length: {df_results['description_length'].mean():.0f} chars")

In [ ]:
# Show sample of results (without full description)
display(df_results[['table_name', 'indicator_id', 'records', 'countries', 'year_range', 'density', 'is_sparse', 'has_recent_data']].head(20))

In [ ]:
# Show a sample description
if len(results) > 0:
    sample = results[0]
    print(f"=" * 60)
    print(f"SAMPLE DESCRIPTION for: {sample['table_name']}")
    print(f"=" * 60)
    print()
    print(sample['proposed_description'])
    print()
    print(f"Length: {sample['description_length']} chars")

In [ ]:
# Show a few more sample descriptions for different data profiles
print("\n" + "=" * 60)
print("SAMPLE: A sparse table")
print("=" * 60)
sparse_tables = [r for r in results if r['is_sparse']]
if sparse_tables:
    print(sparse_tables[0]['proposed_description'])

print("\n" + "=" * 60)
print("SAMPLE: A table without recent data")
print("=" * 60)
old_tables = [r for r in results if not r['has_recent_data']]
if old_tables:
    print(old_tables[0]['proposed_description'])

In [ ]:
# Save results to a Delta table for review
df_spark = spark.createDataFrame(df_results)
output_table = f"{CATALOG}.{SCHEMA}_metadata.description_proposals"

# Create metadata schema if it doesn't exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}_metadata")

df_spark.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(output_table)

print(f"Saved {len(df_results)} description proposals to {output_table}")

---

## Apply Descriptions (After Review)

After reviewing the proposals above, run the cell below to apply the descriptions to the actual tables.

In [ ]:
# Set to True to apply descriptions after review
APPLY_DESCRIPTIONS = False

if APPLY_DESCRIPTIONS:
    print("Applying descriptions to tables...")
    
    success_count = 0
    error_count = 0
    
    for row in tqdm(results, desc="Applying descriptions"):
        table_name = row['table_name']
        description = row['proposed_description']
        full_table_name = f"{CATALOG}.{SCHEMA}.{table_name}"
        
        # Escape single quotes for SQL
        description_escaped = description.replace("'", "''")
        
        try:
            spark.sql(f"COMMENT ON TABLE {full_table_name} IS '{description_escaped}'")
            success_count += 1
        except Exception as e:
            print(f"Error updating {table_name}: {e}")
            error_count += 1
    
    print(f"\nDone! Updated {success_count} tables, {error_count} errors")
else:
    print("APPLY_DESCRIPTIONS is False - descriptions not applied.")
    print("Set APPLY_DESCRIPTIONS = True and re-run this cell to apply.")